# 数据准备和生成模型的Pipeline

In [1]:
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import  Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# 导入数据
filename = 'pima_data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
data = read_csv(filename, names=names)
# 将数据分为输入数据和输出结果
array = data.values
X = array[:, 0:8]
Y = array[:, 8]
num_folds = 10
seed = 7
kfold = KFold(n_splits=num_folds, shuffle=True, random_state=seed)


steps = []
# 创建Pipeline
steps.append(('Standardize', StandardScaler()))
steps.append(('lda', LinearDiscriminantAnalysis()))
model = Pipeline(steps)
result = cross_val_score(model, X, Y, cv=kfold)

print(result.mean())

0.7669685577580315


# 特征选择和生成模型的Pipeline

In [2]:
import warnings
warnings.filterwarnings('ignore')

from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import FeatureUnion
from sklearn.pipeline import  Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest

# 导入数据
filename = 'pima_data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
data = read_csv(filename, names=names)
# 将数据分为输入数据和输出结果
array = data.values
X = array[:, 0:8]
Y = array[:, 8]
num_folds = 10
seed = 7
kfold = KFold(n_splits = num_folds, shuffle = True, random_state = seed)

# 生成 feature union
features = []
features.append(('pca', PCA()))
features.append(('select_best', SelectKBest(k = 6)))
# 生成 Pipeline
steps = []
steps.append(('feature_union', FeatureUnion(features)))
steps.append(('logistic', LogisticRegression()))
model = Pipeline(steps)
result = cross_val_score(model, X, Y, cv = kfold)

print(result.mean())

0.773462064251538


# 补充：更加复杂的pipeline

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer  # 做数据填充
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer # 列差异化处理
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 加载数据集（本地文件路径）
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']
data = pd.read_csv(url, names=column_names)


# 数据预处理：将0值替换为NaN（除了Pregnancies和Outcome）
zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
data[zero_columns] = data[zero_columns].replace(0, np.nan)

# 划分数据集
X = data.drop('Outcome', axis=1)
y = data['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



# Pipeline1:创建特征转换器
numeric_features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Age']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 添加特征工程：创建BMI类别特征
class BmiTransformer:
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        X_copy['BMI_Category'] = pd.cut(
            X_copy['BMI'],
            bins=[0, 18.5, 25, 30, 100],
            labels=['Underweight', 'Normal', 'Overweight', 'Obese']
        )
        return X_copy


# Pipeline2:分类特征处理
categorical_features = ['BMI_Category'] # 新增列BMI_Category，把BMI数值列转换为类别列
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # 众数填充缺失值
    ('onehot', OneHotEncoder(handle_unknown='ignore'))    # 忽略新类别
])

# 完整的列转换器
preprocessor = ColumnTransformer(  # 列差异化处理
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

# 创建完整Pipeline
pipeline = Pipeline(steps=[
    ('feature_engineer', BmiTransformer()),  # 自定义特征工程，这里创建了新的列BMI_Category
    ('preprocessor', preprocessor),          # 预处理，列转换器对原来的列和新增的列分别做处理
    ('classifier', RandomForestClassifier(   # 分类器，试用随机森林建模
        n_estimators=100,
        max_depth=5,
        random_state=42,
        class_weight='balanced'
    ))
])



# 训练模型
pipeline.fit(X_train, y_train)

# 预测评估
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

Accuracy: 0.7403
F1 Score: 0.6774
AUC-ROC: 0.8176
